[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-8_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 8: A Value-Aligned and Ethical Evaluation Framework

### Purpose and scope

This notebook evaluates a small set of candidate vehicle routes using ethical principles, stakeholder priorities, and explicit thresholds. It illustrates how a routing decision can be assessed alongside economic efficiency, environmental impact, worker welfare, customer service, and social equity.

The computational method is a transparent, weighted scoring procedure applied to three supplied alternatives. It does **not** search for routes, solve a vehicle-routing optimization model, or learn ethical principles from feedback. The routes and their operational metrics are illustrative scenario inputs; the metrics are not calculated from the distance matrix. The distance matrix and demand data define a separate small VRP with Backhauls instance and are used to check that the listed route sequences respect the delivery-before-pickup convention and vehicle capacity.

### Modeling assumptions

- Customers indexed from 1 through 4 are delivery customers; customers indexed from 5 through 7 are pickup customers. A candidate route visits every customer once, serves all deliveries before any pickups, and starts and ends at depot 0.
- The vehicle begins with the total delivery load. Delivery demand reduces the load; pickup demand increases it. The supplied routes are checked against the vehicle capacity using this load convention.
- Candidate route metrics such as cost, emissions, working hours, and service quality are supplied scenario values. They are not inferred from travel distances or demands.
- Principle scores are normalized to the interval $[0,1]$ using the explicit transformations in the code. Stakeholder utility is a weighted average of the principle scores prioritized by that stakeholder.
- Constraint violations reduce the aggregate score through a weighted penalty. This is a decision-support illustration, not a validated measure of ethical quality or a substitute for legal, labor, or environmental assessment.
- No demographic, geographic, or time-series data are provided. Consequently, this notebook cannot establish whether a route is biased or whether stakeholder preferences change over time.

### Evaluation method

Let $A$ be the set of supplied alternatives, $S$ the set of stakeholders, and $P$ the set of ethical principles. For alternative $a$, let $q_{ap}\in[0,1]$ be its score on principle $p$. Stakeholder $s$ assigns priority weight $w_{sp}$ to principle $p$, and has overall weight $v_s$. The stakeholder utility and aggregate utility are

$$u_{as}=\frac{\sum_{p\in P}w_{sp}q_{ap}}{\sum_{p\in P}w_{sp}},\qquad U_a=\sum_{s\in S}v_su_{as}.$$

For a constraint $c$, let $r_{ac}\geq 0$ be its normalized violation severity and $\lambda_c$ its penalty weight. The reported score is

$$E_a=\max\left(0,\ U_a-\frac{\sum_c\lambda_cr_{ac}}{\sum_c\lambda_c}\right).$$

Here, $q_{ap}$ is computed by `evaluate_route_ethically`, $u_{as}$ is stored in `stakeholder_utilities`, and $E_a$ is `overall_ethical_score`. Upper-bound and lower-bound constraints use normalized excess beyond their threshold; an equality constraint uses normalized absolute deviation. An alternative with a positive violation is reported as violating that constraint, even though its score is still calculated. The selected alternative is the supplied candidate with the largest reported score; ties are retained when identifying the maximum.

### Workflow

1. Define the VRP with Backhauls instance, ethical principles, stakeholders, and thresholds.
2. Validate the supplied routes against customer coverage, backhaul order, and capacity.
3. Score each candidate against the stated principles and constraints.
4. Select the highest-scoring supplied alternative, record an audit entry, and visualize the comparisons.

The score depends on the chosen normalization, stakeholder priorities, and penalty weights. It is therefore a transparent summary of these assumptions, not an objective or universal ranking of ethical decisions.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
class EthicalPrinciple(Enum):
    """Ethical dimensions used to evaluate candidate routes."""

    ENVIRONMENTAL_SUSTAINABILITY = "environmental_sustainability"
    WORKER_WELFARE = "worker_welfare"
    CUSTOMER_SATISFACTION = "customer_satisfaction"
    ECONOMIC_EFFICIENCY = "economic_efficiency"
    SOCIAL_EQUITY = "social_equity"
    TRANSPARENCY = "transparency"
    ACCOUNTABILITY = "accountability"


@dataclass
class VRPBInstance:
    """Data for a single-vehicle VRP with deliveries followed by pickups."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self):
        self.num_customers = self.num_deliveries + self.num_pickups
        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class Stakeholder:
    """A stakeholder with principle priorities and an aggregate weight."""

    stakeholder_id: str
    stakeholder_type: str
    priorities: Dict[EthicalPrinciple, float]
    utility_function: str
    weight: float
    feedback_history: List[Dict[str, Any]] = field(default_factory=list)


@dataclass
class EthicalConstraint:
    """A threshold used to assess one route metric."""

    constraint_id: str
    principle: EthicalPrinciple
    constraint_type: str
    threshold: float
    penalty_weight: float
    description: str
    satisfaction_level: float = 1.0


@dataclass
class EthicalAuditTrail:
    """A record of the alternatives considered and the selected decision."""

    audit_id: str
    timestamp: datetime
    decision_context: str
    considered_alternatives: List[Dict[str, Any]]
    ethical_evaluation: Dict[str, float]
    final_decision: Dict[str, Any]
    constraint_violations: List[str]
    stakeholder_impact: Dict[str, float]
    justification: str
    reviewer_id: str


@dataclass
class BiasDetection:
    """A bias assessment record; this notebook has no data to populate one."""

    bias_type: str
    detection_method: str
    bias_score: float
    affected_groups: List[str]
    mitigation_strategy: str
    mitigation_effectiveness: float


print("Ethical framework data structures defined.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the small delivery-and-pickup instance used for route checks."""

    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ]
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(
    f"Created an instance with {instance.num_deliveries} delivery customers "
    f"and {instance.num_pickups} pickup customers."
)
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
class ConstitutionalAI:
    """Score routes using declared principles, stakeholder weights, and thresholds."""

    def __init__(self, instance: VRPBInstance):
        self.instance = instance
        self.ethical_constraints = []
        self.stakeholders = {}
        self.audit_trail = []
        self.bias_detections = []
        self.constitution = self._initialize_constitution()
        self._initialize_stakeholders()
        self._initialize_ethical_constraints()

    def _initialize_constitution(self) -> Dict[EthicalPrinciple, Dict[str, Any]]:
        """Set the principle descriptions and aggregate principle weights."""
        return {
            EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: {
                "description": "Limit environmental impact and emissions.",
                "weight": 0.25,
            },
            EthicalPrinciple.WORKER_WELFARE: {
                "description": "Support safe and fair working conditions.",
                "weight": 0.20,
            },
            EthicalPrinciple.CUSTOMER_SATISFACTION: {
                "description": "Support service quality and reasonable waiting times.",
                "weight": 0.20,
            },
            EthicalPrinciple.ECONOMIC_EFFICIENCY: {
                "description": "Consider operating cost and profit margin.",
                "weight": 0.15,
            },
            EthicalPrinciple.SOCIAL_EQUITY: {
                "description": "Consider the distribution of service quality.",
                "weight": 0.10,
            },
            EthicalPrinciple.TRANSPARENCY: {
                "description": "Consider the reported transparency metric.",
                "weight": 0.05,
            },
            EthicalPrinciple.ACCOUNTABILITY: {
                "description": "Consider the reported traceability metric.",
                "weight": 0.05,
            },
        }

    def _initialize_stakeholders(self):
        """Define stakeholder principle priorities and stakeholder weights."""
        stakeholders_config = [
            {
                "stakeholder_id": "CUSTOMER",
                "stakeholder_type": "customer",
                "priorities": {
                    EthicalPrinciple.CUSTOMER_SATISFACTION: 0.4,
                    EthicalPrinciple.ECONOMIC_EFFICIENCY: 0.3,
                    EthicalPrinciple.SOCIAL_EQUITY: 0.2,
                    EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: 0.1,
                },
                "utility_function": "service_quality_cost_balance",
                "weight": 0.3,
            },
            {
                "stakeholder_id": "DRIVER",
                "stakeholder_type": "employee",
                "priorities": {
                    EthicalPrinciple.WORKER_WELFARE: 0.5,
                    EthicalPrinciple.ECONOMIC_EFFICIENCY: 0.3,
                    EthicalPrinciple.SOCIAL_EQUITY: 0.2,
                },
                "utility_function": "welfare_fairness_balance",
                "weight": 0.25,
            },
            {
                "stakeholder_id": "ENVIRONMENT",
                "stakeholder_type": "society",
                "priorities": {
                    EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: 0.6,
                    EthicalPrinciple.SOCIAL_EQUITY: 0.3,
                    EthicalPrinciple.TRANSPARENCY: 0.1,
                },
                "utility_function": "environmental_impact_minimizer",
                "weight": 0.2,
            },
            {
                "stakeholder_id": "COMPANY",
                "stakeholder_type": "business",
                "priorities": {
                    EthicalPrinciple.ECONOMIC_EFFICIENCY: 0.4,
                    EthicalPrinciple.CUSTOMER_SATISFACTION: 0.3,
                    EthicalPrinciple.WORKER_WELFARE: 0.2,
                    EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: 0.1,
                },
                "utility_function": "profit_sustainability_balance",
                "weight": 0.15,
            },
            {
                "stakeholder_id": "REGULATOR",
                "stakeholder_type": "government",
                "priorities": {
                    EthicalPrinciple.SOCIAL_EQUITY: 0.3,
                    EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY: 0.3,
                    EthicalPrinciple.WORKER_WELFARE: 0.2,
                    EthicalPrinciple.ACCOUNTABILITY: 0.2,
                },
                "utility_function": "compliance_equity_balance",
                "weight": 0.1,
            },
        ]

        for config in stakeholders_config:
            stakeholder = Stakeholder(**config)
            self.stakeholders[stakeholder.stakeholder_id] = stakeholder

    def _initialize_ethical_constraints(self):
        """Set operational thresholds and their penalty weights."""
        self.ethical_constraints = [
            EthicalConstraint(
                "ENV_MAX_EMISSIONS",
                EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY,
                "upper_bound",
                100.0,
                2.0,
                "Maximum CO2 emissions per route",
            ),
            EthicalConstraint(
                "WORK_MAX_HOURS",
                EthicalPrinciple.WORKER_WELFARE,
                "upper_bound",
                8.0,
                1.5,
                "Maximum working hours per driver",
            ),
            EthicalConstraint(
                "CUST_MAX_WAIT",
                EthicalPrinciple.CUSTOMER_SATISFACTION,
                "upper_bound",
                4.0,
                1.0,
                "Maximum customer wait time",
            ),
            EthicalConstraint(
                "ECON_MIN_MARGIN",
                EthicalPrinciple.ECONOMIC_EFFICIENCY,
                "lower_bound",
                0.15,
                0.5,
                "Minimum profit margin",
            ),
            EthicalConstraint(
                "SOC_EQUITY_ACCESS",
                EthicalPrinciple.SOCIAL_EQUITY,
                "equality",
                0.2,
                1.0,
                "Target service variance across areas",
            ),
        ]

    @staticmethod
    def _clip_score(value: float) -> float:
        """Keep normalized principle scores within their declared range."""
        return float(np.clip(value, 0.0, 1.0))

    def evaluate_route_ethically(
        self, route: List[int], metrics: Dict[str, float]
    ) -> Dict[str, Any]:
        """Calculate principle scores, stakeholder utilities, and penalties."""
        required_metrics = {
            "operating_cost",
            "co2_emissions",
            "driver_hours",
            "driver_fatigue",
            "max_wait_time",
            "service_quality",
            "profit_margin",
            "service_variance",
            "fuel_efficiency",
            "decision_transparency",
            "decision_traceability",
        }
        missing_metrics = required_metrics.difference(metrics)
        if missing_metrics:
            raise ValueError(f"Missing route metrics: {sorted(missing_metrics)}")

        principle_scores = {
            EthicalPrinciple.ENVIRONMENTAL_SUSTAINABILITY.value: self._clip_score(
                0.5 * (1.0 - metrics["co2_emissions"] / 100.0)
                + 0.5 * metrics["fuel_efficiency"]
            ),
            EthicalPrinciple.WORKER_WELFARE.value: self._clip_score(
                0.5 * (1.0 - metrics["driver_hours"] / 8.0)
                + 0.5 * (1.0 - metrics["driver_fatigue"])
            ),
            EthicalPrinciple.CUSTOMER_SATISFACTION.value: self._clip_score(
                0.5 * metrics["service_quality"]
                + 0.5 * (1.0 - metrics["max_wait_time"] / 4.0)
            ),
            EthicalPrinciple.ECONOMIC_EFFICIENCY.value: self._clip_score(
                0.5 * (metrics["profit_margin"] / 0.30)
                + 0.5 * (150.0 / (150.0 + metrics["operating_cost"]))
            ),
            EthicalPrinciple.SOCIAL_EQUITY.value: self._clip_score(
                1.0 - metrics["service_variance"]
            ),
            EthicalPrinciple.TRANSPARENCY.value: self._clip_score(
                metrics["decision_transparency"]
            ),
            EthicalPrinciple.ACCOUNTABILITY.value: self._clip_score(
                metrics["decision_traceability"]
            ),
        }

        stakeholder_utilities = {}
        for stakeholder_id, stakeholder in self.stakeholders.items():
            priority_total = sum(stakeholder.priorities.values())
            stakeholder_utilities[stakeholder_id] = sum(
                weight * principle_scores[principle.value]
                for principle, weight in stakeholder.priorities.items()
            ) / priority_total

        stakeholder_weight_total = sum(
            stakeholder.weight for stakeholder in self.stakeholders.values()
        )
        aggregate_utility = sum(
            stakeholder.weight * stakeholder_utilities[stakeholder_id]
            for stakeholder_id, stakeholder in self.stakeholders.items()
        ) / stakeholder_weight_total

        metric_by_constraint = {
            "ENV_MAX_EMISSIONS": "co2_emissions",
            "WORK_MAX_HOURS": "driver_hours",
            "CUST_MAX_WAIT": "max_wait_time",
            "ECON_MIN_MARGIN": "profit_margin",
            "SOC_EQUITY_ACCESS": "service_variance",
        }
        violations = []
        weighted_penalty = 0.0
        total_penalty_weight = sum(
            constraint.penalty_weight for constraint in self.ethical_constraints
        )

        for constraint in self.ethical_constraints:
            value = metrics[metric_by_constraint[constraint.constraint_id]]
            scale = max(abs(constraint.threshold), 1e-12)
            if constraint.constraint_type == "upper_bound":
                severity = max(0.0, value - constraint.threshold) / scale
            elif constraint.constraint_type == "lower_bound":
                severity = max(0.0, constraint.threshold - value) / scale
            elif constraint.constraint_type == "equality":
                severity = abs(value - constraint.threshold) / scale
            else:
                raise ValueError(
                    f"Unsupported constraint type: {constraint.constraint_type}"
                )

            weighted_penalty += constraint.penalty_weight * severity
            constraint.satisfaction_level = self._clip_score(1.0 - severity)
            if severity > 1e-12:
                violations.append(
                    {
                        "constraint_id": constraint.constraint_id,
                        "severity": severity,
                        "value": value,
                        "threshold": constraint.threshold,
                    }
                )

        penalty = weighted_penalty / total_penalty_weight
        overall_score = self._clip_score(aggregate_utility - penalty)

        return {
            "route": list(route),
            "principle_scores": principle_scores,
            "stakeholder_utilities": stakeholder_utilities,
            "aggregate_utility": aggregate_utility,
            "constraint_penalty": penalty,
            "overall_ethical_score": overall_score,
            "violations": violations,
            "bias_detections": [],
        }


print("Constitutional evaluation and threshold checks are ready.")

In [ ]:
class ValueAlignedOptimizer:
    """Evaluate and rank the supplied candidate routes by their ethical scores."""

    def __init__(self, instance: VRPBInstance, constitutional_ai: ConstitutionalAI):
        self.instance = instance
        self.constitutional_ai = constitutional_ai
        self.optimization_history = []
        self.stakeholder_feedback = defaultdict(list)

    def generate_ethical_alternatives(self) -> List[Dict[str, Any]]:
        """Attach ethical evaluations to the three supplied scenario alternatives."""
        candidate_data = [
            {
                "route_id": "A",
                "route": [1, 2, 3, 4, 5, 6, 7],
                "description": "Economic-efficiency-focused scenario",
                "focus": "economic_efficiency",
                "metrics": {
                    "total_distance": 25.0,
                    "operating_cost": 150.0,
                    "co2_emissions": 85.0,
                    "driver_hours": 7.5,
                    "driver_fatigue": 0.7,
                    "max_wait_time": 3.5,
                    "service_quality": 0.85,
                    "profit_margin": 0.22,
                    "service_variance": 0.25,
                    "fuel_efficiency": 0.75,
                    "decision_transparency": 0.8,
                    "decision_traceability": 0.9,
                },
            },
            {
                "route_id": "B",
                "route": [1, 3, 2, 4, 6, 5, 7],
                "description": "Environmental- and welfare-focused scenario",
                "focus": "environmental_sustainability",
                "metrics": {
                    "total_distance": 27.0,
                    "operating_cost": 162.0,
                    "co2_emissions": 51.0,
                    "driver_hours": 6.8,
                    "driver_fatigue": 0.4,
                    "max_wait_time": 2.8,
                    "service_quality": 0.92,
                    "profit_margin": 0.18,
                    "service_variance": 0.15,
                    "fuel_efficiency": 0.88,
                    "decision_transparency": 0.9,
                    "decision_traceability": 0.95,
                },
            },
            {
                "route_id": "C",
                "route": [1, 4, 2, 3, 5, 7, 6],
                "description": "Balanced scenario",
                "focus": "balanced",
                "metrics": {
                    "total_distance": 26.0,
                    "operating_cost": 156.0,
                    "co2_emissions": 68.0,
                    "driver_hours": 7.2,
                    "driver_fatigue": 0.55,
                    "max_wait_time": 3.2,
                    "service_quality": 0.88,
                    "profit_margin": 0.20,
                    "service_variance": 0.20,
                    "fuel_efficiency": 0.82,
                    "decision_transparency": 0.85,
                    "decision_traceability": 0.92,
                },
            },
        ]

        alternatives = []
        for candidate in candidate_data:
            alternative = dict(candidate)
            alternative["ethical_evaluation"] = (
                self.constitutional_ai.evaluate_route_ethically(
                    candidate["route"], candidate["metrics"]
                )
            )
            alternatives.append(alternative)

        return alternatives


print("Value-aligned candidate evaluation is ready.")

In [ ]:
constitutional_ai = ConstitutionalAI(instance)
optimizer = ValueAlignedOptimizer(instance, constitutional_ai)

print("\nVALUE-ALIGNED EVALUATION FRAMEWORK")
print("=" * 48)

print("\nEthical principles:")
for principle, config in constitutional_ai.constitution.items():
    print(
        f"  {principle.value.replace('_', ' ').title()}: "
        f"weight={config['weight']:.2f}; {config['description']}"
    )

print("\nStakeholders:")
for stakeholder_id, stakeholder in constitutional_ai.stakeholders.items():
    top_priority = max(stakeholder.priorities, key=stakeholder.priorities.get)
    print(
        f"  {stakeholder_id} ({stakeholder.stakeholder_type}): "
        f"weight={stakeholder.weight:.2f}; "
        f"top priority={top_priority.value.replace('_', ' ')}"
    )

print("\nEthical constraints:")
for constraint in constitutional_ai.ethical_constraints:
    print(
        f"  {constraint.constraint_id}: {constraint.constraint_type} "
        f"{constraint.threshold}; penalty weight={constraint.penalty_weight:.2f}"
    )

In [ ]:
def validate_vrpb_route(route: List[int], instance: VRPBInstance) -> Dict[str, Any]:
    """Check customer coverage, delivery-before-pickup order, and vehicle load."""
    expected = set(instance.all_customer_indices)
    route_is_valid = (
        len(route) == instance.num_customers
        and len(set(route)) == len(route)
        and set(route) == expected
    )
    if not route_is_valid:
        return {
            "valid": False,
            "coverage_valid": False,
            "backhaul_order_valid": False,
            "capacity_valid": False,
            "loads": [],
        }

    first_pickup_position = next(
        (
            position
            for position, customer in enumerate(route)
            if customer in instance.pickup_indices
        ),
        len(route),
    )
    backhaul_order_valid = all(
        customer in instance.delivery_indices
        for customer in route[:first_pickup_position]
    ) and all(
        customer in instance.pickup_indices
        for customer in route[first_pickup_position:]
    )

    load = float(sum(instance.delivery_demands))
    loads = [load]
    capacity_valid = load <= instance.capacity
    for customer in route:
        if customer in instance.delivery_indices:
            load -= instance.delivery_demands[customer - 1]
        else:
            pickup_position = customer - instance.num_deliveries - 1
            load += instance.pickup_demands[pickup_position]
        loads.append(load)
        capacity_valid = capacity_valid and 0 <= load <= instance.capacity

    return {
        "valid": backhaul_order_valid and capacity_valid,
        "coverage_valid": True,
        "backhaul_order_valid": backhaul_order_valid,
        "capacity_valid": capacity_valid,
        "loads": loads,
    }


alternatives = optimizer.generate_ethical_alternatives()

print("ROUTE EVALUATION")
print("=" * 48)
for alternative in alternatives:
    route_check = validate_vrpb_route(alternative["route"], instance)
    if not route_check["valid"]:
        raise ValueError(f"Route {alternative['route_id']} fails the VRPB checks.")

    evaluation = alternative["ethical_evaluation"]
    metrics = alternative["metrics"]
    print(f"\nRoute {alternative['route_id']}: {alternative['description']}")
    print(f"  Customer sequence: {alternative['route']}")
    print(
        f"  Scenario cost: ${metrics['operating_cost']:.0f}; "
        f"emissions: {metrics['co2_emissions']:.1f} kg CO2"
    )
    print(
        f"  Driver hours: {metrics['driver_hours']:.1f}; "
        f"maximum wait: {metrics['max_wait_time']:.1f} hours"
    )
    print(
        f"  Aggregate utility: {evaluation['aggregate_utility']:.3f}; "
        f"constraint penalty: {evaluation['constraint_penalty']:.3f}"
    )
    print(f"  Ethical evaluation score: {evaluation['overall_ethical_score']:.3f}")
    print("  Stakeholder utilities:")
    for stakeholder_id, utility in evaluation["stakeholder_utilities"].items():
        print(f"    {stakeholder_id}: {utility:.3f}")
    if evaluation["violations"]:
        print("  Threshold violations:")
        for violation in evaluation["violations"]:
            print(
                f"    {violation['constraint_id']}: "
                f"value={violation['value']:.3f}, "
                f"threshold={violation['threshold']:.3f}, "
                f"normalized severity={violation['severity']:.3f}"
            )
    else:
        print("  All configured thresholds are satisfied.")
    print("  Bias assessment: not performed; relevant group data are unavailable.")

In [ ]:
scores = [
    alternative["ethical_evaluation"]["overall_ethical_score"]
    for alternative in alternatives
]
best_score = max(scores)
tolerance = 1e-12
best_alternatives = [
    alternative
    for alternative in alternatives
    if abs(alternative["ethical_evaluation"]["overall_ethical_score"] - best_score)
    <= tolerance
]
selected_alternative = best_alternatives[0]
ethical_eval = selected_alternative["ethical_evaluation"]

audit_entry = EthicalAuditTrail(
    audit_id=f"VRPB-{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}",
    timestamp=datetime.utcnow(),
    decision_context=(
        "Selection among three supplied VRPB route scenarios using the "
        "declared stakeholder weights, principle scores, and thresholds."
    ),
    considered_alternatives=[
        {
            "route_id": alternative["route_id"],
            "route": alternative["route"],
            "score": alternative["ethical_evaluation"]["overall_ethical_score"],
        }
        for alternative in alternatives
    ],
    ethical_evaluation=ethical_eval["principle_scores"],
    final_decision={
        "route_id": selected_alternative["route_id"],
        "route": selected_alternative["route"],
        "score": ethical_eval["overall_ethical_score"],
    },
    constraint_violations=[
        violation["constraint_id"] for violation in ethical_eval["violations"]
    ],
    stakeholder_impact=ethical_eval["stakeholder_utilities"],
    justification=(
        "Selected because it has the highest calculated score among the "
        "supplied alternatives under the stated model assumptions."
    ),
    reviewer_id="not_specified",
)
constitutional_ai.audit_trail.append(audit_entry)

print("VALUE-ALIGNED DECISION")
print("=" * 48)
print(f"Selected route: {selected_alternative['route_id']}")
print(f"Description: {selected_alternative['description']}")
print(f"Customer sequence: {selected_alternative['route']}")
print(f"Ethical evaluation score: {ethical_eval['overall_ethical_score']:.3f}")
if len(best_alternatives) > 1:
    tied_ids = [alternative["route_id"] for alternative in best_alternatives]
    print(f"The maximum is tied within tolerance by routes: {tied_ids}")
print(f"Audit record: {audit_entry.audit_id}")
print(
    "The decision ranks only the supplied candidates; it does not establish "
    "a globally optimal route."
)

In [ ]:
def visualize_ethical_framework(
    alternatives: List[Dict[str, Any]],
    selected: Dict[str, Any],
    constitutional_ai: ConstitutionalAI,
):
    """Plot candidate scores, stakeholder utilities, principles, and trade-offs."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("Value-Aligned Route Evaluation", fontsize=16, fontweight="bold")

    route_ids = [alternative["route_id"] for alternative in alternatives]
    ethical_scores = [
        alternative["ethical_evaluation"]["overall_ethical_score"]
        for alternative in alternatives
    ]
    colors = [
        "lightcoral" if alternative["route_id"] == selected["route_id"] else "skyblue"
        for alternative in alternatives
    ]
    bars = ax1.bar(route_ids, ethical_scores, color=colors)
    ax1.set_ylabel("Ethical evaluation score (unitless)")
    ax1.set_title("Score by Candidate Route")
    ax1.set_ylim(0, 1)
    for bar, score in zip(bars, ethical_scores):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{score:.3f}",
            ha="center",
            va="bottom",
        )

    stakeholder_ids = list(constitutional_ai.stakeholders)
    utility_matrix = np.array(
        [
            [
                alternative["ethical_evaluation"]["stakeholder_utilities"][stakeholder_id]
                for alternative in alternatives
            ]
            for stakeholder_id in stakeholder_ids
        ]
    )
    image = ax2.imshow(utility_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax2.set_xticks(range(len(route_ids)), labels=route_ids)
    ax2.set_yticks(range(len(stakeholder_ids)), labels=stakeholder_ids)
    ax2.set_xlabel("Candidate route")
    ax2.set_ylabel("Stakeholder")
    ax2.set_title("Stakeholder Utility")
    colorbar = fig.colorbar(image, ax=ax2)
    colorbar.set_label("Utility score (unitless)")

    selected_evaluation = selected["ethical_evaluation"]
    economic_alternative = next(
        alternative
        for alternative in alternatives
        if alternative["focus"] == "economic_efficiency"
    )
    economic_evaluation = economic_alternative["ethical_evaluation"]
    principle_names = list(selected_evaluation["principle_scores"])
    selected_scores = [
        selected_evaluation["principle_scores"][name] for name in principle_names
    ]
    economic_scores = [
        economic_evaluation["principle_scores"][name] for name in principle_names
    ]
    x_positions = np.arange(len(principle_names))
    width = 0.35
    ax3.bar(
        x_positions - width / 2,
        selected_scores,
        width,
        label=f"Selected route {selected['route_id']}",
        color="lightgreen",
    )
    ax3.bar(
        x_positions + width / 2,
        economic_scores,
        width,
        label="Route A",
        color="lightcoral",
    )
    ax3.set_xlabel("Ethical principle")
    ax3.set_ylabel("Principle score (unitless)")
    ax3.set_title("Principle Scores")
    ax3.set_xticks(
        x_positions,
        labels=[name.replace("_", "\n").title() for name in principle_names],
        rotation=35,
        ha="right",
    )
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    for alternative in alternatives:
        metrics = alternative["metrics"]
        is_selected = alternative["route_id"] == selected["route_id"]
        ax4.scatter(
            metrics["operating_cost"],
            metrics["co2_emissions"],
            s=100,
            color="red" if is_selected else "blue",
            label="Selected" if is_selected else "Other candidate",
        )
        ax4.annotate(
            alternative["route_id"],
            (metrics["operating_cost"], metrics["co2_emissions"]),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax4.set_xlabel("Scenario operating cost ($)")
    ax4.set_ylabel("Scenario emissions (kg CO2)")
    ax4.set_title("Cost and Emissions Trade-off")
    ax4.grid(True, alpha=0.3)
    handles, labels = ax4.get_legend_handles_labels()
    unique_legend = dict(zip(labels, handles))
    ax4.legend(unique_legend.values(), unique_legend.keys())

    fig.tight_layout()
    plt.show()


visualize_ethical_framework(alternatives, selected_alternative, constitutional_ai)

## Summary and Conclusions

### What this implementation demonstrates

- A small VRP with Backhauls instance distinguishes delivery customers from pickup customers and checks complete customer coverage, delivery-before-pickup order, and vehicle capacity for each supplied route.
- A seven-principle framework makes the evaluation dimensions explicit: environmental sustainability, worker welfare, customer satisfaction, economic efficiency, social equity, transparency, and accountability.
- Five stakeholder groups assign different priorities to those principles. Their utilities are combined using the stakeholder weights defined in the code.
- Five thresholds are assessed using upper-bound, lower-bound, or equality tests. Normalized violation severity contributes to a weighted penalty, and violations are reported separately from the score.
- The selected route is the highest-scoring alternative among the three supplied candidates under these specific assumptions. The result is not a proof of global optimality and is not a route found by a routing optimizer.
- The audit record stores the alternatives, scores, selected route, reported violations, stakeholder utilities, timestamp, and decision rationale. It documents the computation; it does not independently validate the inputs or establish accountability by itself.

### Interpreting the scenario

The supplied scenario metrics assign Route A a cost of $150 and emissions of 85 kg CO2, while Route B has a cost of $162 and emissions of 51 kg CO2. These values imply that Route B's cost is 8% higher than Route A's, and its emissions are 40% lower than Route A's. These comparisons describe the supplied metrics only; they are not derived from the distance matrix. The score-based selection also depends on worker, customer, economic, equity, transparency, and accountability measures, as well as the specified thresholds and weights.

### Limitations and responsible use

The score transformations and weights are modeling choices. Different defensible choices can change the ranking, and the score should not be interpreted as a universal ethical measure. The scenario does not provide evidence for demographic bias detection, stakeholder feedback learning, regulatory compliance, or real-world driver and customer satisfaction. Such conclusions require suitable data, defined measurement procedures, and human review. The evaluation also considers only the three listed candidates and does not generate or optimize routes.

### Possible extensions within this framework

The same evaluation structure can be applied to additional candidate routes, additional operational thresholds, or carefully documented stakeholder priorities. Any such extension should state its data sources, units, normalization rules, and limitations, and should preserve a reviewable record of how the reported decision was obtained.